In [ ]:
#Download the needed packages.
import wrds
import pyarrow
import os
import pandas as pd
from pathlib import Path

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [ ]:
db = wrds.Connection() #achieve connection
print("Connected")

## Explore WRDS

In [ ]:
# db.list_libraries() #check access to libs

db.list_tables('ciq_transcripts')

In [ ]:
db.describe_table('ciq_transcripts', 'ciqtranscript')

In [ ]:
db.describe_table('ciq_transcripts', 'ciqtranscript')


In [ ]:
db.describe_table('ciq_transcripts', 'wrds_transcript_detail')

In [ ]:
db.describe_table('ciq_transcripts', 'ciqtranscriptcomponent')

In [ ]:
db.describe_table('ciq_transcripts', 'ciqtranscriptcomponenttype')

In [ ]:
db.describe_table('ciq_transcripts','wrds_transcript_person')

In [ ]:
db.describe_table('ciq_transcripts','ciqtranscriptperson') 

In [ ]:
db.describe_table('ciq_transcripts','ciqtranscriptspeakertype')  

## Query WRDS

In [ ]:
sp500 = pd.read_csv(DATA_PROCESSED/"sp500_link_table.csv")

In [ ]:
company_ids = tuple(sp500["companyid"])


#company_ids_sql = "(" + ",".join(map(str, company_ids)) + ")"
#company_ids_sql

In [ ]:
sp500_ids = set(sp500["companyid"].astype("int64"))
company_ids_sql = "(" + ",".join(map(str, sp500_ids)) + ")"

In [ ]:
company_ids_sql

In [ ]:
batch_size = 500
offset = 0
all_ids = []

while True:

    query = f"""
    SELECT DISTINCT transcriptid
    FROM ciq_transcripts.wrds_transcript_detail
    WHERE keydeveventtypeid = 48
      AND mostimportantdateutc BETWEEN '2020-01-01' AND '2024-12-31'
      AND companyid IN {company_ids_sql}
    ORDER BY transcriptid
    LIMIT {batch_size}
    OFFSET {offset}
    """

    batch = db.raw_sql(query)

    if batch.empty:
        break

    all_ids.append(batch)

    offset += batch_size
    print(f"Fetched {offset} transcripts...")

calls = pd.concat(all_ids, ignore_index=True)

calls.to_csv(DATA_RAW / "calls.csv", index=False)

ids = calls["transcriptid"].astype(int).tolist()

print(f"Total transcripts collected: {len(ids)}")

In [ ]:
meta = db.raw_sql(f"""
SELECT DISTINCT
    transcriptid,
    companyid,
    companyname,
    mostimportantdateutc,
    transcriptcreationdate_utc,
    transcriptcreationtime_utc                  
FROM ciq_transcripts.wrds_transcript_detail
WHERE transcriptid IN {tuple(ids)}
""")

In [ ]:
ids_sql = "(" + ",".join(map(str, ids)) + ")"

text = db.raw_sql(f"""
SELECT
    c.transcriptid,
    c.transcriptcomponentid,
    c.componentorder,
    c.transcriptcomponenttypeid,
    p.transcriptcomponenttypename,
    p.transcriptpersonid,
    p.transcriptpersonname,
    p.proid,
    p.companyofperson,
    p.speakertypeid,
    p.speakertypename,
    p.componenttextpreview,
    p.word_count,
    c.componenttext
FROM ciq_transcripts.ciqtranscriptcomponent c
LEFT JOIN ciq_transcripts.wrds_transcript_person p
    ON c.transcriptcomponentid = p.transcriptcomponentid
WHERE c.transcriptid IN {ids_sql}
  AND c.transcriptcomponenttypeid IN (3,4)
  AND p.speakertypeid IN (2,3)
ORDER BY c.transcriptid, c.componentorder
""")

text.to_csv(DATA_RAW / "text_qna.csv", index=False)

In [ ]:
docs = (
    text.groupby("transcriptid")
    .apply(lambda x: "\n".join(
        f"{row.speakertypename}: {row.componenttext}"
        for _, row in x.iterrows()
    ))
)

In [ ]:
print("Attempting to close connection ...... ")
try: 
    db.close()
    del db
    print("Succes: Connection closed and the object is now removed, feel free to log off.")
except:
    print("Failed: Connection is already closed.")

In [ ]:
docs_df = docs.reset_index(name="transcript_text")
docs_df = docs_df.merge(meta, on="transcriptid", how="left")
docs_df["filepath"] = docs_df["transcriptid"].apply(
    lambda x: f"transcripts/{x}.txt"
)

In [ ]:
print(docs_df)

In [ ]:
docs_df["mostimportantdateutc"] = pd.to_datetime(docs_df["mostimportantdateutc"])

docs_df["quarter"] = docs_df["mostimportantdateutc"].dt.to_period("Q")

docs_df["transcriptcreationdatetime_utc"] = pd.to_datetime(
    docs_df["transcriptcreationdate_utc"].astype(str) + " " + docs_df["transcriptcreationtime_utc"].astype(str)
)

company_docs = (
    docs_df.groupby(["companyid","quarter"])["transcript_text"]
    .apply(" ".join)
)



### Filter Duplicates

In [ ]:
docs_df = (
    docs_df
    .sort_values("transcriptcreationdatetime_utc")
    .drop_duplicates(
        subset=["companyid", "mostimportantdateutc"],
        keep="last"
    )
    .reset_index(drop=True)
)

In [ ]:
docs_df.duplicated(["companyid","mostimportantdateutc"]).sum() #check for duplicates


In [ ]:
#save as csv
os.makedirs("data", exist_ok=True)

docs_df.to_csv(DATA_RAW/"transcripts_final.csv", index=False)

In [ ]:
text = pd.read_csv(DATA_RAW/'text_qna.csv')
text_parquet = text.copy()
obj_cols = text_parquet.select_dtypes(include="object").columns

for col in obj_cols:
    text_parquet[col] = text_parquet[col].astype("string")
    

text_parquet.to_parquet(
    "text.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False
)

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(calls)

pq.write_table(
    table,
    "transcripts.parquet",
    compression="snappy"
)

In [ ]:
docs_df.shape
docs_df.columns
docs_df["transcript_text"].str.len().describe()